# Evolution-1 playground
Evolve a four-legged walker, compare runs, and watch the best one. Edit only the SETTINGS cell, then run all cells (Runtime > Run all on Colab).

## 1. Install and load
On Colab this installs the project from GitHub. Locally it just uses the repo folder.

In [ ]:
import os, subprocess, sys

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "evolution-simulation[research] @ git+https://github.com/000wahab000/Evolution"], check=True)
else:  # running from <repo>/notebooks or <repo>
    root = os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else ".")
    sys.path.insert(0, root)
    os.chdir(root)  # results/ always lands in the repo root

import json
from concurrent.futures import ProcessPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML
from matplotlib import animation

from src.model import CreatureBrain
from src.research.evaluate import fixed_scenarios, from_flat, make_env
from src.research.playground import evolve, make_cfg

## 2. Settings
The only cell you need to change.

In [ ]:
CPG = True            # True: the brain gets a rhythm clock as input. False: no clock (cpg_mode "zeros").
MUTATION_RATE = 0.15  # chance each of the 324 weights is changed when breeding (0.05 = gentle, 0.4 = wild)
SEEDS = [0, 1]        # one independent evolution per seed; more seeds = more trustworthy average
GENERATIONS = 15      # how many rounds of breeding (more = better walkers, slower)
POPULATION = 24       # walkers per generation
RUN_NAME = "my_run"   # name of this run; the result is saved to results/playground/<RUN_NAME>.json

## 3. Run the experiment
Each seed evolves walkers with the repo's genetic algorithm, then tests the best one on fresh terrains it never trained on. Seeds run in parallel. With the default settings this took about 2 minutes (2 seeds, 8 CPUs; each seed takes that long on one core), so expect roughly 2-5 minutes on Colab's 2 CPUs.

In [ ]:
jobs = [(CPG, MUTATION_RATE, GENERATIONS, POPULATION, s) for s in SEEDS]
with ProcessPoolExecutor(min(len(SEEDS), os.cpu_count() or 1)) as pool:
    results = list(pool.map(evolve, jobs))
best = [r[0] for r in results]
champion = results[int(np.argmax(best))][1]
settings = dict(CPG=CPG, MUTATION_RATE=MUTATION_RATE, SEEDS=SEEDS, GENERATIONS=GENERATIONS,
                POPULATION=POPULATION, RUN_NAME=RUN_NAME)
os.makedirs("results/playground", exist_ok=True)
with open(f"results/playground/{RUN_NAME}.json", "w") as f:
    json.dump({"settings": settings, "best_distance_per_seed": dict(zip(map(str, SEEDS), best)),
               "champion_genome": champion}, f)
print("best distance per seed:", dict(zip(SEEDS, np.round(best, 1))))

## 4. Compare runs
Each bar is the average best distance (test terrains) of one saved run; dots are the individual seeds. Your current run is orange.

In [ ]:
runs = []
for name in sorted(os.listdir("results/playground")):
    if name.endswith(".json"):
        with open(f"results/playground/{name}") as f:
            runs.append((name[:-5], list(json.load(f)["best_distance_per_seed"].values())))
fig, ax = plt.subplots(figsize=(1.5 + 1.2 * len(runs), 4))
for i, (name, d) in enumerate(runs):
    ax.bar(i, np.mean(d), color="tab:orange" if name == RUN_NAME else "tab:blue")
    ax.scatter([i] * len(d), d, color="black", zorder=3, s=15)
ax.set_xticks(range(len(runs)), [n for n, _ in runs], rotation=30, ha="right")
ax.set_ylabel("distance walked")
plt.tight_layout()
plt.show()

## 5. Watch the best walker
Replays the best genome of this run on one test terrain and animates it (side view; the brown line is the ground).

In [ ]:
cfg = make_cfg(CPG, SEEDS[0])
scn = fixed_scenarios(cfg, "test")[0]
env = make_env(cfg, scn)
brain = CreatureBrain.from_state_dict(from_flat(np.array(champion)))
state, frames, done = env.reset(), [], False
while not done and len(frames) < cfg.max_steps:
    state, _, done, _ = env.step(brain.get_action(state))
    s = env.get_render_state()
    frames.append((s["torso"], s["legs"]))

xs = [f[0]["x"] for f in frames]
fig, ax = plt.subplots(figsize=(8, 3))
terrain_x = np.linspace(min(xs) - 200, max(xs) + 300, 200)
ground = [scn.terrain.ground_top(x) for x in terrain_x]
ax.plot(terrain_x, [g if g is not None else np.nan for g in ground], color="saddlebrown")
ax.set_xlim(min(xs) - 150, max(xs) + 250)
ax.set_ylim(620, 330)  # y points down in the simulation
ax.set_aspect("equal")
(body,) = ax.plot([], [], "k", lw=6)
legs_ln = [ax.plot([], [], "tab:red", lw=3)[0] for _ in frames[0][1]]


def draw(i):
    t, legs = frames[i]
    c, s = np.cos(t["angle"]), np.sin(t["angle"])
    body.set_data([t["x"] - 30 * c, t["x"] + 30 * c], [t["y"] - 30 * s, t["y"] + 30 * s])
    for ln, lg in zip(legs_ln, legs):
        h = lg["height"] / 2
        a = lg["angle"]
        ln.set_data([lg["x"] + h * np.sin(a), lg["x"] - h * np.sin(a)],
                    [lg["y"] - h * np.cos(a), lg["y"] + h * np.cos(a)])
    ax.set_xlim(t["x"] - 200, t["x"] + 300)
    return [body, *legs_ln]


anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=50, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())